# August 2026 verification · **08 — Findings: every claim against the primary source**

One table. Every claim the article brief rests on, from the BMA's
compilation, Healthcare Management, or the brief itself, sits next to:

* the **rebuilt** figure, read from my Gold tables (never typed in);
* its **basis**: reported-provider (my primary measure, the Two Maps measure), headline (incl. NHS England's estimates for missing trusts), provider-basis CWT, or a sourced estimate;
* a **verdict** from a fixed vocabulary;
* the **Gold file** the figure comes from, and the **primary file** behind it.

**Verdicts**

| verdict | meaning |
|---|---|
| **holds** | reproduced from the primary files on my primary measure (or the only measure that exists) |
| **holds — headline basis only** | exact on NHS England's headline basis; a different number on my measure |
| **true but misleading** | the number is right; the inference drawn from it isn't |
| **overstated** | the data supports a weaker version of the claim |
| **does not hold** | contradicted by the primary files |
| **sourced, not rebuilt** | true to NHS England's publication, but no published file lets me rebuild it |
| **not tested** | outside what these files can check |

This notebook decides nothing new. Every number comes from Gold 00 and
05-07, plus two cells read directly from the Overview workbook for the
completed-pathway medians.

In [1]:
import re
import sys
from pathlib import Path

import openpyxl
import pandas as pd
import pypdf

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, GOLD_DIR, SILVER_DIR  # noqa: E402

OUT = GOLD_DIR / "08_findings"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

G00, G05, G06, G07 = (GOLD_DIR / d for d in ("00_reconciliation", "05_divergence", "06_pathways_patients", "07_cancer"))
PUB = "data/bronze/nhse_rtt_published_2026-08/"
F_OVERVIEW = PUB + "RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Aug26-XLS-139K-vjgRdG.xlsx"
F_PROVIDER = PUB + "Incomplete-Provider-Aug26-XLSX-9M-vjgRdG.xlsx"
F_SPN = PUB + "Aug26-RTT-statistical-press-notice-PDF-389K-vjgRdG.pdf"
F_ADJ = PUB + "Missing-data-adjustments-to-Aug26-XLSX-109K-vjgRdG.xlsx"
F_CSV = "data/bronze/nhse_rtt_2025-04 … nhse_rtt_2026-08/Full-CSV-data-file-*.zip"
F_WLMDS = "data/bronze/nhse_wlmds_2026-08/WLMDS-Summary-to-30-August-2026.xlsx"
F_CWT = "data/bronze/nhse_cwt_2026-08/2026-27-Apr-Aug-Cumulative-Monthly-Combined-CSV-Provisional.csv"
F_CWT_NTS = "data/bronze/nhse_cwt_2026-08/CWT-CRS-National-Time-Series-Oct-2009-Aug-2026-Provisional.xlsx"
F_CWT25 = "data/bronze/nhse_cwt_2025-08_provisional/ (Aug 2025 as first published)"
F_REL = "data/bronze/nhse_cwt_2026-08/Cancer-Waiting-Times-Statistical-Release-August-2026-Provider-based-Provisional.pdf"

## Step 0 — Every upstream stage must be present, and the gate must have passed

In [2]:
gate = pd.read_csv(G00 / "reconciliation.csv")
assert gate["pass"].all(), f"reconciliation gate has {(~gate['pass']).sum()} failing checks: run 04 first"
need = [G00 / "headline_rebuilt.csv", G00 / "claims_vs_primary.csv", G05 / "divergence_summary.csv",
        G05 / "three_row_table.csv", G05 / "march_revision_effect.csv", G05 / "march_targets.csv",
        G06 / "pathways_patients_by_month.csv", G06 / "multi_pathway_bounds.csv",
        G07 / "vs_standard.csv", G07 / "year_on_year.csv", G07 / "provider_spread.csv"]
missing = [p.name for p in need if not p.exists()]
assert not missing, f"run notebooks 04-07 first; missing {missing}"
print(f"gate: {len(gate)} of {len(gate)} checks passed; all {len(need)} upstream tables present")

H = pd.read_csv(G00 / "headline_rebuilt.csv", index_col="period")
CV = pd.read_csv(G00 / "claims_vs_primary.csv").set_index("claim")
DV = pd.read_csv(G05 / "divergence_summary.csv").set_index(["window", "basis"])
T3 = pd.read_csv(G05 / "three_row_table.csv").set_index(["basis", "measure"])
REV = pd.read_csv(G05 / "march_revision_effect.csv").set_index("period")
TGT = pd.read_csv(G05 / "march_targets.csv", index_col=0)
PP = pd.read_csv(G06 / "pathways_patients_by_month.csv", index_col="period")
BND = pd.read_csv(G06 / "multi_pathway_bounds.csv").set_index("max_pathways_per_person_K")
VS = pd.read_csv(G07 / "vs_standard.csv").set_index(["measure", "period"])
YY = pd.read_csv(G07 / "year_on_year.csv").set_index(["measure", "comparison"])
SP = pd.read_csv(G07 / "provider_spread.csv").set_index(["measure", "period"])
GT = gate.set_index("check")

gate: 51 of 51 checks passed; all 11 upstream tables present


### Two figures read straight from the Overview workbook

The brief's opening says a patient referred in August "is treated inside
twelve weeks, because the median wait is 12.2". That 12.2 is the median wait
of people **still waiting**, not of people **treated**. NHS England
publishes medians for completed pathways too, so I read those.

In [3]:
ws = openpyxl.load_workbook(BRONZE_DIR / "nhse_rtt_published_2026-08" /
                            "RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Aug26-XLS-139K-vjgRdG.xlsx",
                            read_only=True, data_only=True).worksheets[0]
orows = list(ws.iter_rows(values_only=True))
group, keys = None, []
for g_, h_ in zip(orows[10], orows[11]):
    group = str(g_).strip() if g_ else group
    keys.append(f"{group} | {str(h_).strip()}" if h_ else None)
aug = next(r for r in orows[12:] if hasattr(r[2], "strftime") and r[2].strftime("%Y-%m") == "2026-08")
MED = {k: float(aug[keys.index(f"{k} | Median wait (weeks)")])
       for k in ["Completed admitted (unadjusted) RTT pathways", "Completed non-admitted RTT pathways"]}

# Completed pathways (admitted + non-admitted, with estimates): the source of
# the "4.7 million" figure. NHS England's press office called it a record
# summer ("more than in any previous summer"); the brief called it a record
# quarter. I test both against every window in the series (from 2007).
ka = keys.index("Completed admitted (unadjusted) RTT pathways | No. of pathways (all) with estimates for missing data")
kn = keys.index("Completed non-admitted RTT pathways | No. of pathways (all) with estimates for missing data")
comp = pd.Series({r[2]: pd.to_numeric(r[ka], errors="coerce") + pd.to_numeric(r[kn], errors="coerce")
                  for r in orows[12:] if hasattr(r[2], "strftime")}).dropna()
comp.index = pd.to_datetime(comp.index)
roll3 = comp.rolling(3).sum().dropna()
JJA = roll3.loc["2026-08"].iloc[0]
rank_any3 = int((roll3 > JJA).sum()) + 1
summers = comp[comp.index.month.isin([6, 7, 8])]
summers = summers.groupby(summers.index.year).agg(["sum", "size"])
summers = summers[summers["size"] == 3]["sum"]
higher_windows = roll3[roll3 > JJA]
print(f"Jun-Aug 2026 completed pathways: {JJA:,.0f}; rank among {len(roll3)} three-month windows: {rank_any3}; "
      f"highest Jun-Aug since {summers.index.min()}: {summers.idxmax() == 2026}")
MED

Jun-Aug 2026 completed pathways: 4,738,824; rank among 226 three-month windows: 3; highest Jun-Aug since 2008: True


{'Completed admitted (unadjusted) RTT pathways': 12.9692604006163,
 'Completed non-admitted RTT pathways': 8.78644683103918}

## Step 1 — The rows

Each row is built from the tables above. The verdict is set by an explicit
rule next to the numbers, so a reader can see why it is what it is.

In [4]:
rows = []
def row(section, claim, source, rebuilt, basis, verdict, detail, gold, primary):
    rows.append({"section": section, "claim_as_stated": claim, "claimed_by": source, "rebuilt": rebuilt,
                 "basis": basis, "verdict": verdict, "detail": detail, "gold_file": gold, "primary_file": primary})

A, J = H.loc["2026-08"], H.loc["2026-07"]

# --- RTT headline figures ------------------------------------------------------
row("RTT headline", "7.37 million pathways", "BMA; Healthcare Management",
    f"{A.total_est:,} headline; {A.total:,} reported", "headline vs reported",
    "holds — headline basis only" if round(A.total_est / 1e6, 2) == 7.37 else "does not hold",
    f"Exact on NHS England's headline basis (reported + carry-forward for RHQ, RA9, RK9, RNS). "
    f"On my Two Maps measure the list is {A.total:,}, {A.total_est - A.total:,} lower.",
    "00_reconciliation/headline_rebuilt.csv", f"{F_CSV}; {F_ADJ}; {F_OVERVIEW}")

d_h, d_r = A.total_est - J.total_est, A.total - J.total
# Account for the gap between the two bases exactly, trust by trust, from NHS England's adjustments file.
adj = pd.read_excel(BRONZE_DIR / "nhse_rtt_published_2026-08" / "Missing-data-adjustments-to-Aug26-XLSX-109K-vjgRdG.xlsx",
                    sheet_name="Adjustments")
adj = adj[adj["RTT Type"].eq("Incomplete")].assign(period=lambda d: d["Estimated month"].dt.strftime("%Y-%m"))
ea = adj[adj.period == "2026-08"].set_index("Prov Code")["Estimated RTT Pathways"]
ej = adj[adj.period == "2026-07"].set_index("Prov Code")["Estimated RTT Pathways"]
de = ea.sub(ej, fill_value=0)
de = de[de != 0]
assert de.sum() == d_h - d_r, "estimate changes do not account for the gap between the bases"
row("RTT headline", "up 41,848 on July", "Healthcare Management",
    f"{d_h:+,} headline; {d_r:+,} reported", "headline vs reported",
    "holds — headline basis only" if d_h == 41_848 else "does not hold",
    f"Exact on the headline basis. The {d_h - d_r:,} gap between the bases is entirely the change in estimated trusts: "
    + "; ".join(f"{c} {v:+,.0f}" for c, v in de.items())
    + " (Northampton newly estimated; Cora resubmitted in August).",
    "00_reconciliation/headline_rebuilt.csv", f"{F_CSV}; {F_ADJ}")

last5 = H.loc["2026-04":"2026-08"]
prev = H.shift(1).loc["2026-04":"2026-08"]
rises_h = int((last5.total_est > prev.total_est).sum())
rises_r = int((last5.total > prev.total).sum())
row("RTT headline", "the fourth rise in five months", "Healthcare Management (headline)",
    f"{rises_h} of 5 headline; {rises_r} of 5 reported (Apr-Aug 2026)", "headline vs reported",
    "holds — headline basis only" if rises_h == 4 and rises_r != 4 else ("holds" if rises_h == rises_r == 4 else "does not hold"),
    "Month-on-month rises in the list, April to August 2026. On my measure non-submitters break the run.",
    "00_reconciliation/headline_rebuilt.csv", f"{F_CSV}; {F_ADJ}")

row("RTT headline", "~113,000 waiting more than 52 weeks", "BMA",
    f"{A.over_52_est:,} headline; {A.over_52:,} reported", "headline vs reported",
    "holds — headline basis only" if round(A.over_52_est / 1e3) == 113 else "does not hold",
    "Over-52 headline rebuilt exactly by carrying forward each missing trust's base-month 52+ count (gate B).",
    "00_reconciliation/headline_rebuilt.csv", f"{F_CSV}; {F_ADJ}; {F_SPN}")

med = CV.loc["rtt_median_weeks", "primary_reported_basis"]
row("RTT headline", "median wait 12.2 weeks", "BMA",
    f"{med:.2f} weeks", "reported (NHS England publishes no headline-basis median)",
    "holds" if round(med, 1) == 12.2 else "does not hold",
    "Rebuilt by interpolation in the weekly bands; matches the Overview series in all 17 months.",
    "00_reconciliation/claims_vs_primary.csv", f"{F_CSV}; {F_SPN}")

row("RTT headline", "a record 4.7 million treatments, procedures and appointments across June to August", "article brief (NHS England press office wording)",
    f"{JJA:,.0f} completed RTT pathways, Jun-Aug 2026 (with estimates)", "headline",
    "holds" if summers.idxmax() == 2026 and round(JJA / 1e6, 1) == 4.7 else "does not hold",
    f"Rebuilt exactly from the Overview series, and identical to NHS England's press figure. It is the highest of all "
    f"{len(summers)} full June-Augusts in the series ({summers.index.min()}-2026), {JJA - summers.drop(2026).max():,.0f} above {summers.drop(2026).idxmax()}. Wording: these are COMPLETED "
    "RTT PATHWAYS, which include clock stops other than treatment, so 'pathways completed' is the accurate phrase.",
    "— (computed in this notebook)", F_OVERVIEW)

row("RTT headline", "its highest recorded quarter of elective activity", "article brief (premise)",
    f"rank {rank_any3} of {len(roll3)} three-month windows", "headline",
    "does not hold" if rank_any3 > 1 else "holds",
    "A record for a summer, not for any three months: "
    + "; ".join(f"{d:%b %Y}-ending window {v:,.0f}" for d, v in higher_windows.sort_values(ascending=False).items())
    + " were higher.",
    "— (computed in this notebook)", F_OVERVIEW)

# --- the Sheffield sidebar -------------------------------------------------------------
cw = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_provider_headline.parquet")
rhq = cw[(cw.org_code == "RHQ") & (cw.measure == "62D")].set_index("period").sort_index()
rhq_pct = 100 * rhq.within / rhq.total
repeats = int(((rhq.total == rhq.total.shift()) & (rhq.within == rhq.within.shift())).sum())
pooled = cw[(cw.measure == "62D") & cw.period.isin(["2026-06", "2026-07", "2026-08"])].groupby("org_code")[["total", "within"]].sum()
pooled = pooled[pooled.total >= 20].assign(pct=lambda d: 100 * d.within / d.total).sort_values("pct")
rank_rhq = list(pooled.index).index("RHQ") + 1
annex_hits = []
for f in ["nhse_cwt_2026-08/Cancer-Waiting-Times-Statistical-Release-August-2026-Provider-based-Provisional.pdf",
          "nhse_cwt_2025-08_provisional/Cancer-Waiting-Times-Statistical-Release-August-2025-Provider-based-Provisional.pdf"]:
    t = " ".join(p.extract_text() for p in pypdf.PdfReader(BRONZE_DIR / f).pages)
    annex_hits.append(bool(re.search(r"Sheffield Teaching|\(RHQ\)", t)))
ok = len(rhq) == 17 and repeats == 0 and not any(annex_hits) and rank_rhq == 1
row("Sidebar", "Sheffield sends no RTT data, yet is the lowest trust on 62-day cancer (51.8%)", "verification outline",
    f"{pooled.loc['RHQ', 'pct']:.1f}% Jun-Aug 2026 pooled, rank {rank_rhq} of {len(pooled)}; 62-day reported in {len(rhq)} of 17 months",
    "CWT provider basis (reported, never estimated)", "holds" if ok else "does not hold",
    f"Two separate collections. RTT has been withheld since its July 2025 EPR go-live (data-quality issues, per NHS England's notice); "
    f"its cancer return kept coming, with no repeated month ({repeats} identical consecutive months) and no mention in either "
    f"release's missing/partial annex. Its 62-day rate was {rhq_pct.loc['2025-04':'2025-06'].min():.1f}-{rhq_pct.loc['2025-04':'2025-06'].max():.1f}% "
    f"in Apr-Jun 2025, before the RTT gap, and {rhq_pct.min():.1f}-{rhq_pct.max():.1f}% across all 17 months: low throughout, not a post-EPR artefact. "
    "Context to check before use: a trust hosting a specialist cancer centre receives late inter-trust referrals.",
    "— (computed in this notebook from silver/cwt)", f"{F_CWT}; {F_REL}; {F_SPN}")

# --- the shelf-life line ---------------------------------------------------------------
row("Timing", "the 12 November publication replaces it", "article brief",
    "12 November 2026, 9:30am (provisional)", "—", "holds",
    "From gov.uk's RTT announcement for September 2026 (published 9 Oct 2026, read 17 Oct 2026). The brief cited the A&E announcement; cite the RTT one.",
    "—", "https://www.gov.uk/government/statistics/announcements/referral-to-treatment-waiting-times-statistics-for-consultant-led-elective-care-for-september-2026")

# --- the divergence ----------------------------------------------------------------
t52, tl = T3.loc[("headline", "Over 52 weeks")], T3.loc[("headline", "Total list")]
r52 = T3.loc[("reported", "Over 52 weeks")]
row("Divergence", "+7,000 more people waiting over a year (title)", "article brief",
    f"{t52.mom_abs:+,.0f} headline; {r52.mom_abs:+,.0f} reported (Jul→Aug)", "headline vs reported",
    "does not hold" if abs(t52.mom_abs - 7000) > 500 else "holds",
    f"No single month gives +7,000. The nearest span is June→August on the headline basis "
    f"(+{H.loc['2026-08', 'over_52_est'] - H.loc['2026-06', 'over_52_est']:,}), which is two months.",
    "05_divergence/three_row_table.csv", f"{F_CSV}; {F_ADJ}")

row("Divergence", "over-52-week group grew by roughly six per cent", "article brief",
    f"{t52.mom_pct:+.2f}% headline; {r52.mom_pct:+.2f}% reported (Jul→Aug)", "headline vs reported",
    "does not hold" if abs(t52.mom_pct - 6) > 0.5 else "holds",
    f"About 6-7% only over two months (May→Jul {100 * (H.loc['2026-07', 'over_52_est'] / H.loc['2026-05', 'over_52_est'] - 1):+.1f}%, "
    f"Jun→Aug {100 * (H.loc['2026-08', 'over_52_est'] / H.loc['2026-06', 'over_52_est'] - 1):+.1f}%); "
    f"+{DV.loc[('Mar26', 'headline'), 'over_52_change_pct']:.1f}% since March 2026.",
    "05_divergence/three_row_table.csv; 05_divergence/month_on_month.csv", f"{F_CSV}; {F_ADJ}")

row("Divergence", "the list grew by well under one per cent", "article brief",
    f"{tl.mom_pct:+.2f}% headline; {T3.loc[('reported', 'Total list'), 'mom_pct']:+.2f}% reported (Jul→Aug)",
    "headline vs reported",
    "holds" if tl.mom_pct < 1 and T3.loc[("reported", "Total list"), "mom_pct"] < 1 else "does not hold",
    "Holds on both bases.", "05_divergence/three_row_table.csv", f"{F_CSV}; {F_ADJ}")

a52, al = DV.loc[("Apr25", "headline"), "over_52_change_pct"], DV.loc[("Apr25", "headline"), "total_change_pct"]
row("Divergence", "indexed April 2025 = 100: one line near flat, one climbing away", "article brief (chart)",
    f"tail {a52:+.1f}%, list {al:+.1f}% (headline); panel and reported agree", "all three bases",
    "does not hold" if a52 < 0 else "holds",
    "On the proposed index the tail FELL about 40% while the list stayed flat. The chart as briefed would show the opposite of the argument.",
    "05_divergence/divergence_summary.csv; 05_divergence/index_long.csv", f"{F_CSV}; {F_ADJ}")

m52 = {b: DV.loc[("Mar26", b), "over_52_change_pct"] for b in ["reported", "panel", "headline"]}
mtl = {b: DV.loc[("Mar26", b), "total_change_pct"] for b in ["reported", "panel", "headline"]}
faster = all(m52[b] > mtl[b] for b in m52)
row("Divergence", "the total and the tail move at different rates, 'like separate queues'", "article brief",
    f"since Mar 2026: tail {m52['headline']:+.1f}% vs list {mtl['headline']:+.1f}% (headline); "
    f"panel {m52['panel']:+.1f}% vs {mtl['panel']:+.1f}%; reported {m52['reported']:+.1f}% vs {mtl['reported']:+.1f}%",
    "all three bases", "holds" if faster else "does not hold",
    f"Holds from March 2026 on every basis; the tail grew faster in "
    f"{int(DV.loc[('Mar26', 'headline'), 'months_tail_faster'])} of {int(DV.loc[('Mar26', 'headline'), 'months'])} months. "
    "Over 18 weeks tracks the list, not the tail. Before March the tail was shrinking faster than the list.",
    "05_divergence/divergence_summary.csv", f"{F_CSV}; {F_ADJ}")

mr = REV.loc["2026-03"]
t3 = TGT.loc["2026-03"]
row("Divergence", "(implicit) the March 2026 low is a real starting point", "this verification",
    f"revision moved Mar-26 by {mr.revised_total - mr.orig_total:+,.0f} pathways ({int(mr.providers_changed)} trust)",
    "reported", "holds",
    f"Not a revision artefact. A system-wide year-end low at the deadline for the 65% / <1% targets "
    f"(March: {t3.pct_within_18:.1f}% within 18 weeks, met; {t3.pct_over_52:.2f}% over 52, missed), "
    "with elevated long-wait exits and unreported removals. Say so if the window starts there.",
    "05_divergence/march_*.csv; 05_divergence/tail_cohort_exits.csv",
    "superseded + revised Mar26 full CSVs; Overview; 2025/26 planning guidance (web page)")

# --- pathways vs patients -----------------------------------------------------------
pa = PP.loc["2026-08"]
row("Pathways vs patients", "6.24 million patients", "BMA; Healthcare Management",
    f"{pa.patients:,.0f} (ratio {pa.ratio:.4f} × {pa.pathways:,.0f})", "headline × WLMDS ratio",
    "sourced, not rebuilt",
    "An estimate, not a count: the WLMDS ratio of unique NHS numbers to pathways (week ending 30 Aug) times monthly pathways. "
    f"The ratio appears only as a result; the press notice rounds it to 85:100 (which would give {0.85 * pa.pathways / 1e6:.2f}m).",
    "06_pathways_patients/pathways_patients_by_month.csv", f"{F_OVERVIEW}; {F_SPN}; {F_WLMDS}")

row("Pathways vs patients", "~1.1 million people on more than one pathway", "article brief",
    f"{pa.surplus_pathways:,.0f} surplus pathways = upper bound on people with 2+",
    "headline × WLMDS ratio", "overstated",
    f"Exact as 'more pathways than patients'. As people it is a ceiling: if anyone holds 3+ pathways the true count is lower "
    f"(e.g. ≥{BND.loc[3, 'N2_lower_bound']:,.0f} if nobody holds more than 3). No published file gives the distribution.",
    "06_pathways_patients/multi_pathway_bounds.csv", f"{F_OVERVIEW}; {F_WLMDS}")

row("Pathways vs patients", "most coverage reports the pathway figure as if it were people", "article brief",
    "—", "—", "not tested",
    "A claim about the media, not the data. Needs a review of coverage. NHS England's own notice says 'Some patients are on multiple pathways'.",
    "—", F_SPN)

# --- the opening vignette ---------------------------------------------------------
row("Opening", "referred in August 2026 … one joins the 113,000 waiting more than a year", "article brief",
    f"a pathway starting in Aug 2026 is at most {31 / 7:.1f} weeks old at month end", "logic",
    "does not hold",
    "Nobody referred in August 2026 can be in the over-52-week count for August 2026. The vignette needs someone referred in or before August 2025.",
    "—", "definition (RTT clock start)")

row("Opening", "treated inside twelve weeks, because the median wait is 12.2", "article brief",
    f"12.2 = median of people still waiting; completed medians Aug-26: admitted "
    f"{MED['Completed admitted (unadjusted) RTT pathways']:.1f} wks, non-admitted {MED['Completed non-admitted RTT pathways']:.1f} wks",
    "reported", "true but misleading",
    "The 12.2 weeks is a snapshot of the waiting stock, not the time to treatment for a new referral. "
    "For the time people actually waited, use the completed-pathway medians.",
    "—", F_OVERVIEW)

# --- cancer ---------------------------------------------------------------------
c62, cfd = VS.loc[("62D", "2026-08")], VS.loc[("FDS", "2026-08")]
y62 = YY.loc[("62D", "vs Aug 2025 provisional (like for like)")]
yfd = YY.loc[("FDS", "vs Aug 2025 provisional (like for like)")]
sp62, spfd = SP.loc[("62D", "Jun-Aug 2026")], SP.loc[("FDS", "Jun-Aug 2026")]
row("Cancer", "62-day performance 71.9 per cent", "BMA",
    f"{c62.pct:.2f}%", "CWT provider basis, provisional", "holds" if round(c62.pct, 1) == 71.9 else "does not hold",
    f"Provisional: August 2025 moved {YY.loc[('62D', 'Aug 2025 revision (provisional -> final)'), 'change_pp']:+.2f} pp when finalised.",
    "07_cancer/vs_standard.csv", f"{F_CWT}; {F_REL}; {F_CWT_NTS}")
row("Cancer", "against an 85 per cent standard", "article brief",
    f"{c62.standard_pct:.0f}%", "NHS England time-series header", "holds",
    "'Operational Standard = 85% from Q3 2023/24'.", "07_cancer/vs_standard.csv", F_CWT_NTS)
row("Cancer", "improved year on year", "article brief",
    f"{y62.change_pp:+.2f} pp like for like ({y62.aug_2025_pct:.1f}% → {y62.aug_2026_pct:.1f}%); "
    f"{YY.loc[('62D', 'vs Aug 2025 final (mixed vintages)'), 'change_pp']:+.2f} pp vs final",
    "CWT provider basis", "holds" if y62.change_pp > 0 else "does not hold",
    "Like for like = provisional Aug 2025 (first published 9 Oct 2025) against provisional Aug 2026.",
    "07_cancer/year_on_year.csv", f"{F_CWT}; {F_CWT25}")
row("Cancer", "still thirteen points short", "article brief",
    f"{c62.gap_pp:.1f} pp; {c62.shortfall_people:,.0f} more on time needed", "CWT provider basis",
    "holds" if round(c62.gap_pp) == 13 else "does not hold",
    f"Only {int(sp62.meeting_standard)} of {int(sp62['trusts_n>=20'])} trusts meet 85% (Jun-Aug 2026 pooled); "
    f"{int(sp62.clearly_below_standard)} are clearly below it.",
    "07_cancer/vs_standard.csv; 07_cancer/provider_spread.csv", F_CWT)
row("Cancer", "28-day faster diagnosis 78.6 per cent", "BMA",
    f"{cfd.pct:.2f}%", "CWT provider basis, provisional", "holds" if round(cfd.pct, 1) == 78.6 else "does not hold",
    "Provisional.", "07_cancer/vs_standard.csv", f"{F_CWT}; {F_REL}; {F_CWT_NTS}")
row("Cancer", "against an 80 per cent operational target", "article brief",
    f"{cfd.standard_pct:.0f}%", "NHS England time-series header", "holds",
    "'Operational Standard = 80% from Q1 2026/27'.", "07_cancer/vs_standard.csv", F_CWT_NTS)
row("Cancer", "a target missed by under two points", "article brief",
    f"{cfd.gap_pp:.2f} pp", "CWT provider basis", "holds" if cfd.gap_pp < 2 else "does not hold",
    f"Year on year {yfd.change_pp:+.2f} pp like for like; {int(spfd.meeting_standard)} of {int(spfd['trusts_n>=20'])} trusts now meet 80%.",
    "07_cancer/vs_standard.csv; 07_cancer/provider_spread.csv", F_CWT)
row("Cancer", "still means tens of thousands told late", "article brief",
    f"{cfd.late:,.0f} late; {cfd.late_but_allowed:,.0f} permitted by the 80% standard; {cfd.shortfall_people:,.0f} beyond it",
    "CWT provider basis", "true but misleading",
    f"The count is right, but most of it is what the standard allows. The miss itself is {cfd.shortfall_people:,.0f} people "
    f"in August, close to 62-day's {c62.shortfall_people:,.0f}.",
    "07_cancer/vs_standard.csv", F_CWT)

# --- method ---------------------------------------------------------------------
row("Method", "figures consistent with my earlier RTT pieces (Two Maps measure)", "my standard",
    f"Jul-26 {int(GT.loc['Jul26 total = Two Maps', 'rebuilt']):,} / {int(GT.loc['Jul26 over 52 = Two Maps', 'rebuilt']):,}",
    "reported", "holds",
    "This pipeline reproduces Two Maps' July figures exactly. Any headline-basis figure quoted must be labelled as such.",
    "00_reconciliation/reconciliation.csv", f"{F_CSV}; {F_PROVIDER}")

row("Method", "basis for national figures in the article", "decision, 17 Oct 2026",
    "headline basis (NHS England's, incl. estimates for non-submitting trusts)", "headline", "holds",
    "National England figures are quoted on NHS England's headline basis. Two Maps used the reported basis because it joined "
    "provider-level data, where estimates can't be allocated to trusts. The since-March divergence holds on headline, reported and constant-panel bases (row on 'separate queues').",
    "05_divergence/divergence_summary.csv", F_OVERVIEW)

findings = pd.DataFrame(rows)
findings.insert(0, "id", range(1, len(findings) + 1))
findings.to_csv(OUT / "findings.csv", index=False)
findings[["id", "section", "claim_as_stated", "rebuilt", "verdict"]]

,id,section,claim_as_stated,rebuilt,verdict
0,1,RTT headline,7.37 million pathways,"7,370,100 headline; 7,158,164 reported",holds — headline basis only
1,2,RTT headline,"up 41,848 on July","+41,848 headline; +26,479 reported",holds — headline basis only
2,3,RTT headline,the fourth rise in five months,4 of 5 headline; 3 of 5 reported (Apr-Aug 2026),holds — headline basis only
3,4,RTT headline,"~113,000 waiting more than 52 weeks","113,339 headline; 109,936 reported",holds — headline basis only
4,5,RTT headline,median wait 12.2 weeks,12.21 weeks,holds
5,6,RTT headline,"a record 4.7 million treatments, procedures and appointments across June to ...","4,738,824 completed RTT pathways, Jun-Aug 2026 (with estimates)",holds
6,7,RTT headline,its highest recorded quarter of elective activity,rank 3 of 226 three-month windows,does not hold
7,8,Sidebar,"Sheffield sends no RTT data, yet is the lowest trust on 62-day cancer (51.8%)","51.8% Jun-Aug 2026 pooled, rank 1 of 136; 62-day reported in 17 of 17 months",holds
8,9,Timing,the 12 November publication replaces it,"12 November 2026, 9:30am (provisional)",holds
9,10,Divergence,"+7,000 more people waiting over a year (title)","+1,960 headline; +1,673 reported (Jul→Aug)",does not hold


## Step 2 — Tally

In [5]:
order = ["holds", "holds — headline basis only", "true but misleading", "overstated", "does not hold",
         "sourced, not rebuilt", "not tested"]
assert set(findings.verdict) <= set(order), set(findings.verdict) - set(order)
tally = findings.verdict.value_counts().reindex(order, fill_value=0)
tally

verdict
holds                          16
holds — headline basis only     4
true but misleading             2
overstated                      1
does not hold                   5
sourced, not rebuilt            1
not tested                      1
Name: count, dtype: int64

## Step 3 — A readable copy

The same table as Markdown, for reading next to the draft. The CSV is the
record; this is a convenience copy, regenerated on every run.

In [6]:
md = ["# August 2026 RTT / CWT — claims against the primary source", "",
      "Generated by `notebooks/08_findings.ipynb` from Gold 00 and 05-07. Do not edit by hand.", ""]
for sec, g in findings.groupby("section", sort=False):
    md += [f"## {sec}", "", g[["id", "claim_as_stated", "rebuilt", "basis", "verdict", "detail"]]
           .to_markdown(index=False), ""]
md += ["## Tally", "", tally.rename("claims").to_frame().to_markdown(), ""]
(OUT / "findings.md").write_text("\n".join(md))
# Relative path, so the committed output carries no machine-specific location.
print(f"{len(findings)} claims → {(OUT / 'findings.csv').relative_to(GOLD_DIR.parents[1])} and findings.md")

30 claims → data/gold/08_findings/findings.csv and findings.md
